# BERT をファインチューニングして日本語の固有表現抽出器を作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_ner/bert_ner_finetune.ipynb)

NLP クックブック 20-11 ／ 固有表現抽出

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

日本語の固有表現抽出モデルを、文字位置のアノテーション付きデータから学習したい。

In [1]:
!pip install -q datasets seqeval accelerate

In [2]:
import json, urllib.request
from datasets import Dataset, DatasetDict

# ストックマーク社の Wikipedia 固有表現データセット (CC BY-SA 3.0)
url = "https://raw.githubusercontent.com/stockmarkteam/ner-wikipedia-dataset/main/ner.json"
data = json.load(urllib.request.urlopen(url))
print(len(data), data[0])

types = sorted({e["type"] for d in data for e in d["entities"]})
labels = ["O"] + [f"{p}-{t}" for t in types for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(labels)}
ds = Dataset.from_list([{"text": d["text"], "starts": [e["span"][0] for e in d["entities"]],
                         "ends": [e["span"][1] for e in d["entities"]], "types": [e["type"] for e in d["entities"]]} for d in data])
# テストを先に確保し、残りから開発用を取り分ける（学習・開発・テストは約 70・10・20%）
holdout = ds.train_test_split(test_size=0.2, seed=0)
train_dev = holdout["train"].train_test_split(test_size=0.125, seed=0)
ds = DatasetDict(train=train_dev["train"], dev=train_dev["test"], test=holdout["test"])
print(ds)

5343 {'curid': '3572156', 'text': 'SPRiNGSと最も仲の良いライバルグループ。', 'entities': [{'name': 'SPRiNGS', 'span': [0, 7], 'type': 'その他の組織名'}]}
DatasetDict({
    train: Dataset({
        features: ['text', 'starts', 'ends', 'types'],
        num_rows: 3739
    })
    dev: Dataset({
        features: ['text', 'starts', 'ends', 'types'],
        num_rows: 535
    })
    test: Dataset({
        features: ['text', 'starts', 'ends', 'types'],
        num_rows: 1069
    })
})


In [3]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification

model_name = "xlm-roberta-base"   # 文字位置 (offset mapping) を返せる高速トークナイザを持つモデルを使う
tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode(example):
    enc = tokenizer(example["text"], truncation=True, max_length=256, return_offsets_mapping=True)
    tags = []
    for tok, (s, e) in zip(enc.tokens(), enc["offset_mapping"]):
        if s == e:                                  # 特殊トークンは損失計算から除外
            tags.append(-100)
            continue
        tag = "O"
        if tok == "▁":                              # 単独の「▁」は次の文字と同じ位置を返すので、常に O にする
            tags.append(label2id[tag])
            continue
        for start, end, typ in zip(example["starts"], example["ends"], example["types"]):
            if s < end and e > start:               # トークンがスパンと重なる
                tag = ("B-" if s <= start else "I-") + typ
                break
        tags.append(label2id[tag])
    enc["labels"] = tags
    enc.pop("offset_mapping")
    return enc

tokenized = ds.map(encode, remove_columns=["text", "starts", "ends", "types"])
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label2id)

In [4]:
import numpy as np
from seqeval.metrics import classification_report, f1_score
from transformers import Trainer, TrainingArguments

def to_tags(logits, gold):
    pred = np.argmax(logits, axis=-1)
    y_true = [[labels[g] for g in row if g != -100] for row in gold]
    y_pred = [[labels[p] for p, g in zip(prow, grow) if g != -100] for prow, grow in zip(pred, gold)]
    return y_true, y_pred

def compute_metrics(eval_pred):
    y_true, y_pred = to_tags(*eval_pred)
    return {"f1": f1_score(y_true, y_pred)}

args = TrainingArguments(
    output_dir="ner-out", num_train_epochs=5, learning_rate=5e-5, warmup_steps=60, weight_decay=0.01,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=True, logging_steps=50, report_to="none", seed=0,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["dev"],
                  data_collator=DataCollatorForTokenClassification(tokenizer), compute_metrics=compute_metrics)
trainer.train()
# 開発セットの F1 が最も高かったモデルで、テストセットを 1 回だけ評価する
out = trainer.predict(tokenized["test"])
print(classification_report(*to_tags(out.predictions, out.label_ids), digits=3))

Epoch,Training Loss,Validation Loss,F1
1,0.556113,0.254868,0.654835
2,0.208354,0.209947,0.763732
3,0.120457,0.162536,0.795934
4,0.071752,0.185621,0.803870
5,0.047546,0.179460,0.812407


              precision    recall  f1-score   support

     その他の組織名      0.684     0.744     0.713       215
       イベント名      0.758     0.831     0.793       189
          人名      0.891     0.924     0.907       555
          地名      0.847     0.880     0.864       435
      政治的組織名      0.730     0.801     0.764       226
         施設名      0.744     0.875     0.804       216
         法人名      0.844     0.844     0.844       506
         製品名      0.701     0.815     0.753       270

   micro avg      0.799     0.854     0.825      2612
   macro avg      0.775     0.839     0.805      2612
weighted avg      0.802     0.854     0.827      2612



In [5]:
from transformers import pipeline

ner = pipeline("token-classification", model=trainer.model, tokenizer=tokenizer, aggregation_strategy="simple", device=0)
text = "ソニーグループの吉田憲一郎会長は、東京都港区の本社で新型プレイステーションを発表した。"
for e in ner(text):
    print(text[e["start"]:e["end"]], e["entity_group"], round(float(e["score"]), 2))

ソニーグループ 法人名 0.99
吉田憲一郎 人名 1.0
東京都港区 地名 0.99


- データを学習・開発・テスト（約 70・10・20%）に 3 分割し、各エポックの後に開発セットのエンティティ F1 を計算して最良のチェックポイント（ここでは 5 エポック目、開発 F1 0.812）を復元し、テストセットは最後に 1 回だけ評価した。
- 検証環境の L4 GPU では 5 エポックの学習に約 2 分かかり、テストセットでは micro F1 が 0.825、macro F1 が 0.805（人名 0.907、地名 0.864、法人名 0.844 など）だった。
- ここではサブワードすべてにラベルを付けている（先頭だけに付けて残りを -100 にする流儀もある）。評価は本来、単語や文字の単位に戻して行うのが厳密。
- XLM-R のトークナイザは、文頭や空白の後に単独の `▁` トークンを置き、その文字位置を次の文字と同じ `(0, 1)` のように返す。そのまま位置で判定すると `▁` と次のトークンの両方が `B-` になり、推論時に「ソ」と「ソニーグループ」のように同じ固有表現が二重に出る。
- 日本語特化の BERT (`tohoku-nlp/bert-base-japanese-v3`) のほうが精度は高いが、トークナイザが文字位置を返さないので、位置合わせを自分で実装する必要がある。